# Historical archive: NewFewShotsGemma2 (1)

**Status: not independently reproduced.**

Checkpoint / purpose: `google/gemma-2-2b`.

Historical code, not independently reproduced. Saved settings can differ from report labels; notebook filenames are not authoritative configurations. Exploratory classification and prompting variants. The report describes NaN failures; no verified comparable performance claim is made.

Publication edits only: credentials use `HF_TOKEN`; personal absolute paths were relocated; saved outputs, attachments and runtime metadata were removed. Historical installation cells and local data dependencies are retained. Review code before running; it can download models, install packages and start expensive training. See `docs/experiment_catalogue.md` and the new pipeline for supported experiments.


## Fine-Tunining Gemma-2-2b-it on ECTHR_A using few-shots technique and LoRA

In [ ]:
import torch
import torch.nn as nn
from sklearn.preprocessing import MultiLabelBinarizer
from datasets import Dataset, load_dataset
import pandas as pd
import os
import numpy as np
from peft import TaskType, LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model

from transformers import (
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding)

import bitsandbytes as bnb

import torch
print(torch.cuda.is_available())

In [ ]:
# Ensure proper backend and amp settings
#torch.backends.cudnn.benchmark = True
#torch.backends.cuda.matmul.allow_tf32 = True
#torch.backends.cudnn.allow_tf32 = True

### Defining Article-id, id-Article Mapping

In [ ]:
id_mapping = {
    0: 'Article 1: Obligation to respect human rights',
    1: 'Article 2: Right to life',
    2: 'Article 3: Prohibition of torture',
    3: 'Article 4: Prohibition of slavery and forced labour',
    4: 'Article 5: Right to liberty and security',
    5: 'Article 6: Right to a fair trial',
    6: 'Article 7: No punishment without law',
    7: 'Article 8: Right to respect for private and family life',
    8: 'Article 9: Freedom of thought, conscience and religion',
    9: 'Article 10: Right to freedom of speech and expression',
    10: 'No Articles Violated'
}

article_mapping = {
    'Article 1: Obligation to respect human rights': 0,
    'Article 2: Right to life': 1,
    'Article 3: Prohibition of torture': 2,
    'Article 4: Prohibition of slavery and forced labour': 3,
    'Article 5: Right to liberty and security': 4,
    'Article 6: Right to a fair trial': 5,
    'Article 7: No punishment without law': 6,
    'Article 8: Right to respect for private and family life': 7,
    'Article 9: Freedom of thought, conscience and religion': 8,
    'Article 10: Right to freedom of speech and expression': 9,
    'No Articles Violated': 10
}

### Compute Metrics Function

In [ ]:
import torch
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    
    # Convert logits to predicted labels
    if isinstance(logits, torch.Tensor):
        logits = logits.float()  # Convert logits to float32
        predictions = (logits.sigmoid() > 0.5).cpu().numpy().astype(int)
    else:
        predictions = (1 / (1 + np.exp(-logits)) > 0.5).astype(int)

    # Ensure labels are float32 as well
    if isinstance(labels, torch.Tensor):
        labels = labels.cpu().numpy().astype(np.float32)

    # Calculate metrics
    accuracy = accuracy_score(labels, predictions)
    precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(labels, predictions, average='micro', zero_division=0)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(labels, predictions, average='macro', zero_division=0)

    return {
        "accuracy": accuracy,
        "precision_micro": precision_micro,
        "recall_micro": recall_micro,
        "f1_micro": f1_micro,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro
    }


In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

def sigmoid(x):
    return 1/(1+np.exp(-x))

def compute_metrics(eval_pred):
    
    logits, labels = eval_pred
    predictions = sigmoid(logits)
    predictions = (logits > 0.5).astype(np.float32)
    
    # Calculate accuracy (for multi-label, you may want to use a custom definition of accuracy)
    accuracy = accuracy_score(labels, predictions)
    
    # Calculate micro and macro precision, recall, and F1 scores
    precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(labels, predictions, average='micro')
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(labels, predictions, average='macro')
    
    # Return a dictionary of all the metrics
    return {
        "accuracy": accuracy,
        "precision_micro": precision_micro,
        "recall_micro": recall_micro,
        "f1_micro": f1_micro,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro
    }


### Step 1: Model Loading and Configuration

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Enables 4-bit quantization
    bnb_4bit_use_double_quant=True,  # Use double quantization for potentially higher accuracy (optional)
    bnb_4bit_quant_type="nf4",  # Quantization type (specifics depend on hardware and library)
)

In [ ]:
model_id = "google/gemma-2-2b"
access_token = __import__('os').environ['HF_TOKEN']  

from transformers import AutoConfig, AutoModel

# Create a configuration manually if necessary

#config = AutoConfig.from_pretrained('/teamspace/studios/this_studio/config.json')

model = AutoModelForSequenceClassification.from_pretrained(
    model_id,  
    num_labels=11,  # Number of output labels
    problem_type="multi_label_classification", 
    id2label=id_mapping,  
    label2id=article_mapping,  
    quantization_config=bnb_config,  # configuration for quantization 
    token = access_token,
    use_cache=False,
)

In [ ]:
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=16,  # Reduction factor (lower r means more parameters in the adapter)
    lora_alpha=32,  # Dimensionality of the adapter projection
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj","gate_proj", "up_proj", "down_proj"],  # List of modules to apply the LoRA adapter
    lora_dropout=0.05,  # Dropout rate for the adapter
    bias="none",  # Bias configuration for the adapter
    task_type="SEQ_CLS"  # Task type (sequence classification in this case)
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id, token=access_token, use_fast=True)
print(f' Vocab size of the model {model_id}: {len(tokenizer.get_vocab())}')

### Step 2:  Data Processing

In [ ]:
train = pd.read_parquet('ecthr_a_article_desc_train.parquet')
test = pd.read_parquet('ecthr_a_article_desc_test.parquet')
validation = pd.read_parquet('ecthr_a_article_desc_validation.parquet')


# Convert Pandas DataFrames to Hugging Face Datasets
train_dataset = Dataset.from_pandas(train)
test_dataset = Dataset.from_pandas(test)
validation_dataset = Dataset.from_pandas(validation)

In [ ]:
# Rename the column to match what the model expects
train_dataset = train_dataset.rename_column("labels", "original_labels")
test_dataset = test_dataset.rename_column("labels", "original_labels")
validation_dataset = validation_dataset.rename_column("labels", "original_labels")

In [ ]:
# Function to map articles to their corresponding IDs
def map_articles(articles):
    return [article_mapping[article] for article in articles]

# Apply map_articles function to the 'articles' column
def apply_mapping(examples):
    examples['labels'] = map_articles(examples['articles'])
    return examples

# Apply to each dataset
train_dataset = train_dataset.map(apply_mapping)
test_dataset = test_dataset.map(apply_mapping)
validation_dataset = validation_dataset.map(apply_mapping)


In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

# Initialize the MultiLabelBinarizer with the classes (range from 0 to 10 because there are 11 labels)
mlb = MultiLabelBinarizer(classes=range(11))

# Function to apply one-hot encoding using MultiLabelBinarizer
#def one_hot_encode_labels(examples):
#    one_hot_labels = mlb.fit_transform(examples['labels'])
#    examples['one_hot_labels'] = list(one_hot_labels)
#    return examples

# Function to apply one-hot encoding using MultiLabelBinarizer
def one_hot_encode_labels(examples):
    one_hot_labels = mlb.fit_transform(examples['labels'])  # Use transform instead of fit_transform
    examples['one_hot_labels'] = [label.astype(np.float32) for label in one_hot_labels]  # Convert to float32
    return examples


# Apply to each dataset
train_dataset = train_dataset.map(one_hot_encode_labels, batched=True)
test_dataset = test_dataset.map(one_hot_encode_labels, batched=True)
validation_dataset = validation_dataset.map(one_hot_encode_labels, batched=True)


In [ ]:
type(train_dataset['one_hot_labels'][0])

In [ ]:
# Assuming train_dataset is your dataset object and you want to drop the 'articles' column

train_dataset = train_dataset.remove_columns(["original_labels", "labels"])
test_dataset = test_dataset.remove_columns(["original_labels", "labels"])
validation_dataset = validation_dataset.remove_columns(["original_labels", "labels"])


In [ ]:
# Rename the column to match what the model expects
train_dataset = train_dataset.rename_column("one_hot_labels", "labels")
test_dataset = test_dataset.rename_column("one_hot_labels", "labels")
validation_dataset = validation_dataset.rename_column("one_hot_labels", "labels")

In [ ]:
print(train_dataset.column_names)
print(test_dataset.column_names)
print(validation_dataset.column_names)


In [ ]:

# Define the tokenize function
def tokenize_function(examples):
    inputs = examples['text']
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")
    return model_inputs

# Apply the tokenize function to each dataset
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)
tokenized_validation_dataset = validation_dataset.map(tokenize_function, batched=True)


from datasets import Dataset
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer

def preprocess_function(train, test, validation, tokenizer, instructions, subset_ratio=0.05, seed=42):
    # Function to map articles to their corresponding IDs
    def map_articles(articles):
        return [article_mapping[article] for article in articles]
    
    # Apply the mapping function to each dataset
    train['labels'] = train['articles'].apply(map_articles)
    test['labels'] = test['articles'].apply(map_articles)
    validation['labels'] = validation['articles'].apply(map_articles)

    # Convert labels to a multi-hot encoded format using MultiLabelBinarizer
    mlb = MultiLabelBinarizer(classes=range(11))

    train_labels = mlb.fit_transform(train['labels'])
    test_labels = mlb.transform(test['labels'])
    validation_labels = mlb.transform(validation['labels'])

    # Convert the dataframes to Hugging Face datasets
    train_dataset = Dataset.from_pandas(train[['text']])
    test_dataset = Dataset.from_pandas(test[['text']])
    validation_dataset = Dataset.from_pandas(validation[['text']])

    # Add the multi-hot encoded labels in float format
    train_dataset = train_dataset.add_column('labels', list(train_labels.astype(np.float32)))
    test_dataset = test_dataset.add_column('labels', list(test_labels.astype(np.float32)))
    validation_dataset = validation_dataset.add_column('labels', list(validation_labels.astype(np.float32)))

    # Select a subset of the training dataset
    train_subset = train_dataset.shuffle(seed=seed).select(range(int(len(train_dataset) * subset_ratio)))

    # Tokenize the datasets with instructions
    def tokenize_function(examples):
        inputs = [text for text in examples['text']]
        return tokenizer(inputs, max_length=4096, truncation=True, padding="max_length")

    # Apply tokenization
    tokenized_train_dataset = train_subset.map(tokenize_function, batched=True)
    tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)
    tokenized_validation_dataset = validation_dataset.map(tokenize_function, batched=True)

    return tokenized_train_dataset, tokenized_test_dataset, tokenized_validation_dataset


In [ ]:
training_args = TrainingArguments(
    output_dir="step_weights",
    learning_rate= 1e-6,
    per_device_train_batch_size=6,
    per_device_eval_batch_size=8,
    max_steps=2,
    fp16 = False,
    #num_train_epochs=1,
    logging_steps=5,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    eval_strategy='epoch',
    eval_steps=5,  # Adjusted for more frequent evaluation
    save_strategy="epoch",
    save_steps=15,  # Adjusted for more frequent saving
    load_best_model_at_end=True,
    report_to="none",
    metric_for_best_model = "f1_micro",
    greater_is_better = True,
    optim="adamw_8bit"  # Using a standard optimizer for compatibility
)


In [ ]:
trainer = Trainer(
    model = model,  # The LoRA-adapted model
    args = training_args,  # Training arguments
    train_dataset = tokenized_train_dataset,  # Training dataset 0.2%
    eval_dataset = tokenized_test_dataset,  # Evaluation dataset
    tokenizer = tokenizer,  # Tokenizer for processing text
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer),  # Data collator for preparing batches
    compute_metrics = compute_metrics,  # Function to calculate evaluation metrics
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
def check_sparse_rows(dataset, padding_token_id=0, threshold=0.8):
    """
    Check for sparse rows in the dataset.
    
    Parameters:
    - dataset: The dataset to check.
    - padding_token_id: The token ID used for padding.
    - threshold: The proportion of padding tokens above which a row is considered sparse.
    
    Returns:
    - A list of indices of sparse rows.
    """
    sparse_rows = []
    for i, row in enumerate(dataset):
        input_ids = row['input_ids']
        padding_count = input_ids.count(padding_token_id)
        padding_proportion = padding_count / len(input_ids)
        if padding_proportion > threshold:
            sparse_rows.append(i)
    return sparse_rows


In [ ]:
# Check for sparse rows in the tokenized train dataset
sparse_train_rows = check_sparse_rows(tokenized_train_dataset)
print(f"Sparse rows in train dataset: {len(sparse_train_rows)} / {len(tokenized_train_dataset)}")

# Check for sparse rows in the tokenized test dataset
sparse_test_rows = check_sparse_rows(tokenized_test_dataset)
print(f"Sparse rows in test dataset: {len(sparse_test_rows)} / {len(tokenized_test_dataset)}")

# Check for sparse rows in the tokenized validation dataset
sparse_validation_rows = check_sparse_rows(tokenized_validation_dataset)
print(f"Sparse rows in validation dataset: {len(sparse_validation_rows)} / {len(tokenized_validation_dataset)}")


In [ ]:
def predict_example(text, tokenizer, model, threshold=0.85):
    # Tokenize the text
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=4096)

    # Move input to the same device as model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Make prediction
    with torch.no_grad():
        logits = model(**inputs).logits

    # Apply sigmoid to convert logits to probabilities
    probabilities = torch.sigmoid(logits).squeeze()

    # Apply threshold to convert probabilities to binary predictions
    predictions = (probabilities > threshold).int()

    # Map predicted labels to their names
    predicted_labels = [id_mapping[i] for i, label in enumerate(predictions) if label == 1]

    return predicted_labels, probabilities

# Example usage
example_text  = train['text'][34]
predicted_labels, probabilities = predict_example(example_text, tokenizer, model)
print("Predicted Labels:", predicted_labels)
print("Probabilities:", probabilities)


In [ ]:
train['articles'][34]

In [ ]:
def predict_batch(texts, tokenizer, model, threshold=0.5):
    all_predictions = []
    all_probabilities = []

    for text in texts:
        predicted_labels, probabilities = predict_example(text, tokenizer, model, threshold)
        all_predictions.append(predicted_labels)
        all_probabilities.append(probabilities)  # Ensure this is a list of numbers

    return all_predictions, all_probabilities


In [ ]:
import torch

def predict_example(text, tokenizer, model, threshold=0.5):
    # Tokenize the text
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=4096)

    # Move input to the same device as model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Make prediction
    with torch.no_grad():
        logits = model(**inputs).logits

    # Apply sigmoid to convert logits to probabilities
    probabilities = torch.sigmoid(logits).squeeze().cpu().numpy()  # Convert to numpy array

    # Apply threshold to convert probabilities to binary predictions
    predictions = (probabilities > threshold).astype(int)

    # Map predicted labels to their names
    predicted_labels = [id_mapping[i] for i, label in enumerate(predictions) if label == 1]

    return predicted_labels, probabilities


In [ ]:
from sklearn.metrics import f1_score, recall_score
import numpy as np

def evaluate_model(predictions, ground_truth_labels, threshold=0.5):
    y_true = []
    y_pred = []

    for true_labels, pred_probs in zip(ground_truth_labels, probabilities):
        # Convert true_labels to binary format
        y_true_row = [0] * len(article_mapping)
        for label in true_labels:
            if label in article_mapping:
                y_true_row[article_mapping[label]] = 1
            else:
                print(f"Warning: Label '{label}' not found in article_mapping.")
        y_true.append(y_true_row)

        # Convert predicted probabilities to binary format
        if isinstance(pred_probs, np.ndarray):
            y_pred_row = [1 if prob > threshold else 0 for prob in pred_probs]
        else:
            raise TypeError(f"Expected numpy.ndarray or list, got {type(pred_probs)}")

        y_pred.append(y_pred_row)
    
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # Calculate micro and macro F1 scores
    micro_f1 = f1_score(y_true, y_pred, average='micro')
    macro_f1 = f1_score(y_true, y_pred, average='macro')

    # Calculate micro and macro recall scores
    micro_recall = recall_score(y_true, y_pred, average='micro')
    macro_recall = recall_score(y_true, y_pred, average='macro')

    return micro_f1, macro_f1, micro_recall, macro_recall


In [ ]:
# Test data
test_texts = [train['text'][53]]
test_labels = [list(train['articles'][53])]

# Generate predictions
predictions, probabilities = predict_batch(test_texts, tokenizer, model, threshold=0.8)

# Debugging outputs
print("Predictions:", predictions)
print("Probabilities:", probabilities)
print("Type of probabilities:", type(probabilities[0]))  # Should be numpy.ndarray or list
print("Type of individual probabilities:", type(probabilities[0][0]))  # Should be numpy.float32 or float

# Evaluate the model
micro_f1, macro_f1, micro_recall, macro_recall = evaluate_model(predictions, test_labels, threshold=0.5)

print(f"Micro F1 Score: {micro_f1}")
print(f"Macro F1 Score: {macro_f1}")
print(f"Micro Recall Score: {micro_recall}")
print(f"Macro Recall Score: {macro_recall}")


In [ ]:
print(test_labels)